[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/foucault/foucault.ipynb)

# Foucault's Pendulum

This example computes the turning of Foucault's pendulum. In 1851 Léon Foucault hung a heavy pendulum from the dome of the Panthéon in Paris, and over the hours the line it swung along turned slowly round, clockwise seen from above, though nothing pushed it. The pendulum keeps its swing; the floor turns underneath it, carried by the earth. Only part of the earth's turn lies in the floor, all of it at the pole and none at the equator, and the same part comes out of carrying a direction once around the latitude circle, as the area of the polar cap the circle bounds.

The earth's turn and the floor are modelled as planes, `equator = x ^ y` and `north ^ east`, and the part of the turn the floor feels is a projection of one plane onto the other, `(equator | floor) / floor.squared()`, with no axis of rotation in between. The swing is a direction on the floor, turned back by a rotor in the floor's plane. Carried around the latitude circle, a direction turns at each step by the least rotor taking one site's vertical to the next, `(1 + sites[1:] * sites[:-1]).normalized()`, and the product of those rotors is the holonomy.

In the vector notation of rotating frames the floor feels the component of the earth's angular velocity along the local vertical, $\boldsymbol\Omega\cdot\hat{\mathbf n} = \Omega\sin\varphi$ at latitude $\varphi$, and the swing turns at $-\Omega\sin\varphi$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from functools import reduce

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.mechanics.foucault import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# The earth's space: x and y span the equator, x through the prime meridian, and the axis is z.
x, y, z = mv.basis()                                       # [] Vector each
# The plane the earth turns in.
equator = x ^ y                                            # [] Bivector

# The latitudes: the equator, thirty degrees north, the Panthéon in Paris, and the north pole.
latitudes = np.radians(np.array([0.0, 30.0, 48.85, 90.0]))
paris = 2

## 1. The turn the floor feels

The floor at a site is the plane of its north and east, `north ^ east`. The earth turns in the plane of the equator, and the part of that turn lying in the floor's plane is the inner product of the two bivectors over the floor's own square: a projection, of one plane onto another. It is the sine of the latitude, so all of the earth's turn lies in the floor at the pole and none at the equator. The pendulum swings on, and relative to its floor its line turns back by that part of the earth's turn: clockwise seen from above in the north. Over the day the bob draws a rosette on the floor, coloured here from dawn to dusk: one line at the equator, half a turn at thirty degrees, three quarters of a turn in Paris, and a full turn at the pole.

In [ ]:
# The swings drawn in a day, far fewer than the thousands a real pendulum makes, so that each stays apart,
# and the samples of a day.
n_swings = 40
n_samples = 4000

In [ ]:
def turn(plane: Bivector, angle: np.ndarray) -> Rotor:
    """The turn by an angle in a plane through the origin."""
    return (plane * (-angle / 2)).exp()


def compass(latitudes: np.ndarray) -> tuple[Vector, Vector, Vector]:
    """At a site on the prime meridian at each latitude: its vertical, the direction north along its floor,
    and the direction east."""
    vertical = turn(x ^ z, latitudes) >> x                                    # [...] Vector
    north = turn(x ^ z, latitudes + np.pi / 2) >> x                           # [...] Vector
    return vertical, north, y * np.ones_like(latitudes)                       # [...] Vector each


def felt(floors: Bivector) -> Scalar:
    """How much of the earth's turn lies in each floor's plane: the earth's plane projected onto it, as a
    fraction of the floor's own turn."""
    return (equator | floors) / floors.squared()                       # [...] Scalar


def swing(latitudes: np.ndarray, earth_turns: np.ndarray) -> Vector:
    """The direction a pendulum at each latitude swings along, on its floor, after the earth has turned by
    each angle: started north, and turned back against the floor by the part of the earth's turn the floor
    feels."""
    _, north, east = compass(latitudes)
    # The floor's plane, north towards east.
    floor = north ^ east                                                      # [...] Bivector
    # The turn back by the felt part of the earth's turn: `turn(floor, -felt * earth_turns)`.
    return (floor * felt(floor) * (earth_turns / 2)).exp() >> north           # [...] Vector


_, north, east = compass(latitudes)
print("fraction of the earth's turn each floor feels:", (-render.scalars(felt(north ^ east))).round(4) + 0.0)
turns = np.linspace(0, 2 * np.pi, n_samples)                                     # [samples]
tracks = swing(latitudes[:, None], turns[None, :]) * np.sin(n_swings * turns)   # [latitudes, samples] Vector
save_figure(render.draw_rosettes(tracks, north, east), "foucault");

## 2. The same turn as a holonomy

A direction on the floor, carried once around the latitude circle in small steps, each step the least turn that takes one site's vertical to the next, comes back turned although no step turned it about the vertical. The turn left over is the area of the polar cap the circle bounds on the unit sphere, a full turn less the sine of the latitude's share, measured anticlockwise: and that is exactly the pendulum's turn, clockwise. The pendulum lives through the same transport: a swing that never turns about its own vertical, carried around once a day.

In the notation of differential geometry this is the Gauss–Bonnet theorem: the holonomy around the circle at latitude $\varphi$ is the enclosed area, $2\pi(1 - \sin\varphi)$.

In [ ]:
# The steps a direction is carried around a latitude circle in.
n_steps = 720

In [ ]:
def carried(latitudes: np.ndarray, n_steps: int) -> Rotor:
    """The turn a direction on the floor makes when carried once around each latitude circle, in so many
    steps, each the least turn taking one site's vertical to the next."""
    vertical, _, _ = compass(latitudes)
    sites = turn(equator, 2 * np.pi * np.arange(n_steps + 1)[:, None] / n_steps) >> vertical   # [steps + 1, ...] Vector
    moves = (1 + sites[1:] * sites[:-1]).normalized()                          # [steps, ...] Rotor
    return reduce(lambda total, move: move * total, moves)                    # [...] Rotor


transported = carried(latitudes, n_steps) >> north                               # [latitudes] Vector
day_turns = swing(latitudes, np.full(len(latitudes), 2 * np.pi))               # [latitudes] Vector
print("carried around, turned clockwise by:", np.degrees(render.turned(transported, north, east)).round(2) % 360)
print("the pendulum after a day, turned by: ", np.degrees(render.turned(day_turns, north, east)).round(2) % 360)

## 3. A day in Paris

The earth turns once; the site and its floor ride round the latitude circle, the cap it bounds shaded on the globe, and on the floor the pendulum's line turns back, three quarters of a turn by the end of the day, while the bob's track fans out behind it.

In [ ]:
# The frames of the day in Paris, and the duration of each.
n_frames = 72
duration_ms = 80

In [ ]:
day_turns_paris = np.linspace(0, 2 * np.pi, n_frames + 1)                  # [frames + 1]
vertical, paris_north, paris_east = compass(latitudes[paris])
earth = turn(equator, day_turns_paris)                                         # [frames + 1] Rotor
lines = swing(latitudes[paris], day_turns_paris)                               # [frames + 1] Vector
show_animation(render.animate_day(day_turns_paris, earth, lines, vertical, paris_north, paris_east, tracks[paris]),
               "foucault_day", duration_ms)

In [ ]:
# checks
# After a day each swing has turned clockwise by the sine of its latitude times a full turn, and a direction
# carried around the latitude circle comes back turned the same way, up to the steps it was carried in.
expected = 2 * np.pi * np.sin(latitudes)
assert np.abs(np.sin(render.turned(day_turns, north, east) - expected)).max() < 1e-10
assert np.abs(np.sin(render.turned(transported, north, east) - expected)).max() < 1e-4